# 05 — K-Means Patient Segmentation, Personas & Financial Scenarios

## Objective
1. Standardize core behavioral and value features (`intent_score`, `response_time_hours`, `engagement_score`, `estimated_case_value_inr`).
2. Evaluate cluster counts ($k \in \{2, 3, 4, 5, 6\}$) using Inertia, Silhouette Score, and Davies-Bouldin Index.
3. Profile the 3 operational personas (**High-Risk Disengaged**, **High-Intent Converters**, **High-Value At-Risk**).
4. Quantify drop-off case value exposure and clearly labeled recovery scenarios.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

from src.data_quality import load_csvs
from src.segmentation import (
    SEGMENTATION_FEATURES,
    evaluate_cluster_counts,
    build_persona_profiles,
)
from src.analytics import compute_economic_scenarios

In [2]:
tables = load_csvs(DATA)
pj = tables['patient_journey']

cluster_eval = evaluate_cluster_counts(pj)
print('=== Cluster Quality Evaluation (k = 2..6) ===')
display(cluster_eval)

=== Cluster Quality Evaluation (k = 2..6) ===


,k,inertia,silhouette_score,davies_bouldin_index
0,2,4211.509815,0.258209,1.439413
1,3,3532.437933,0.228358,1.415314
2,4,2995.936157,0.230323,1.307801
3,5,2581.600883,0.230271,1.167377
4,6,2337.594229,0.224473,1.188068


## 1. Operational Persona Profiles ($k=3$)

In [3]:
persona_profiles, pj_personas, km_model = build_persona_profiles(pj, n_clusters=3, random_state=42)
display(persona_profiles)

,cluster,patients,avg_intent_score,avg_response_time_hours,avg_engagement_score,avg_case_value_inr,consultation_rate,completion_rate,dropoff_rate,gross_margin_inr,persona
0,0,503,45.629821,19.764612,32.070775,402099.737575,0.451292,0.147117,0.371769,12737977,High-Risk Disengaged
1,1,571,75.265674,15.033100,69.703853,327004.989492,0.819615,0.602452,0.101576,52199806,High-Intent Converters
2,2,426,63.127700,16.427465,55.486620,716827.232394,0.678404,0.319249,0.246479,46576573,High-Value At-Risk


## 2. Financial Exposure & Recovery Scenario Modeling

In [4]:
scenarios = compute_economic_scenarios(pj)
display(scenarios)

,metric,inr,basis
0,Observed gross revenue,248325245,Observed
1,Observed gross margin,111514356,Observed
2,Drop-off estimated case-value exposure,166511306,"Observed case value, not guaranteed revenue"
3,Top 20% of coded drop-offs by case value,54398190,Observed case value
4,5% recovery scenario,8325565,Scenario: 5% of drop-off case value converts
5,10% recovery scenario,16651131,Scenario: 10% of drop-off case value converts
6,10% recovery scenario margin,7477452,Scenario using 44.9% observed completed-case m...
7,20% recovery scenario,33302261,Scenario: 20% of drop-off case value converts
